1. Activation Functions 
2. Optimizers 
3. Attention Mechanism

**Transformer Attention Mechanism**

In [65]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def self_attention(Q,K,V):
    """
    attention =  softmax((Q.KT/sqrt(K)).V
    """
    scores = softmax((Q @ K.T) / np.sqrt(K.shape[-1]))@V
    return scores
    


In [67]:
Q = np.array([[1, 0], [0, 1]])
K = np.array([[1, 0], [0, 1]])
V = np.array([[1, 2], [3, 4]])

output = self_attention(Q, K, V)
output

array([[1.6604769, 2.6604769],
       [2.3395231, 3.3395231]])

**MultiHead Attention**

In [42]:
import numpy as np

def compute_qkv(X, W_q, W_k, W_v):
    """Computes Query, Key, and Value matrices."""
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    return Q, K, V

def self_attention(Q, K, V):
    """
    Computes scaled dot-product attention for all heads simultaneously.
    Expects input shapes: (n_heads, seq_len, d_k)
    """
    d_k = Q.shape[-1]
    
    # 1. Batched dot-product: (n_heads, seq_len, d_k) @ (n_heads, d_k, seq_len)
    # We use np.swapaxes to transpose only the last two dimensions of K
    scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
    
    # 2. Numerically stable softmax along the last axis (seq_len)
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # 3. Multiply by Value matrix: (n_heads, seq_len, seq_len) @ (n_heads, seq_len, d_k)
    output = attention_weights @ V
    
    return output

def multi_head_attention(Q, K, V, n_heads):
    """
    Vectorized multi-head attention without loops.
    """
    seq_len, d_model = Q.shape
    d_k = d_model // n_heads
    
    # 1. Reshape to split into heads: (seq_len, n_heads, d_k)
    # 2. Transpose to process heads as a batch: (n_heads, seq_len, d_k)
    Q_split = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    K_split = K.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    V_split = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    
    # 3. Compute attention for all heads at once
    # Resulting shape: (n_heads, seq_len, d_k)
    head_outputs = self_attention(Q_split, K_split, V_split)
    
    # 4. Revert the transpose: (seq_len, n_heads, d_k)
    # 5. Reshape (flatten the last two dimensions) back to: (seq_len, d_model)
    final_output = head_outputs.transpose(1, 0, 2).reshape(seq_len, d_model)
    
    return final_output

# --- Test Case ---

X = np.array([[1, 2, 3, 4], 
                [5, 6, 7, 8]])  # (2, 4)

W_q = W_k = W_v = np.eye(4)

Q, K, V = compute_qkv(X, W_q, W_k, W_v)
result = multi_head_attention(Q, K, V, n_heads=2)

print("Vectorized Output Matrix:\n", result)

Vectorized Output Matrix:
 [[4.99917423 5.99917423 6.99999999 7.99999999]
 [5.         6.         7.         8.        ]]


**Masked Self Attention**

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def compute_qkv(X: np.ndarray, W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray):
	"""
	Compute Query (Q), Key (K), and Value (V) matrices.
	"""
	return np.dot(X, W_q), np.dot(X, W_k), np.dot(X, W_v)

def masked_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray) -> np.ndarray:
	"""
	Compute masked self-attention.
	"""
	score = (Q @ K.T) / np.sqrt(K.shape[-1])
	maskedScore = score + mask
	softmax_scores = softmax(maskedScore)@V
	return softmax_scores

In [ ]:
# Example 1 

Q = np.array([[1, 0], [0, 1]])
K = np.array([[1, 0], [0, 1]])
V = np.array([[1, 2], [3, 4]])
mask = np.zeros((Q.shape[0],Q.shape[0]),dtype=np.float16) - np.inf
upper_indices = np.tril_indices(mask.shape[0])
mask[upper_indices] = 0
output = masked_attention(Q, K, V,mask)
output


array([[1.       , 2.       ],
       [2.3395231, 3.3395231]])

In [ ]:
# Example 2 

np.random.seed(42)
X = np.arange(48).reshape(6,8)
X = np.random.permutation(X.flatten()).reshape(6, 8)
W_q = np.random.randint(0,4,size=(8,8))
W_k = np.random.randint(0,5,size=(8,8))
W_v = np.random.randint(0,6,size=(8,8))
Q, K, V = compute_qkv(X, W_q, W_k, W_v)
mask = np.zeros((Q.shape[0],Q.shape[0]),dtype=np.float16) - np.inf
upper_indices = np.tril_indices(mask.shape[0])
mask[upper_indices] = 0
output = masked_attention(Q, K, V,mask)
output

array([[547., 490., 399., 495., 485., 439., 645., 393.],
       [547., 490., 399., 495., 485., 439., 645., 393.],
       [471., 472., 429., 538., 377., 450., 531., 362.],
       [471., 472., 429., 538., 377., 450., 531., 362.],
       [471., 472., 429., 538., 377., 450., 531., 362.],
       [471., 472., 429., 538., 377., 450., 531., 362.]])

### Layer Normalization

In [133]:
def layer_normalization(X,gamma,beta,eps=1e-5):
    # Mean of the features 
    mean = np.mean(X,axis=-1,keepdims=True)
    # Variance of the feature 
    var = np.var(X,axis=-1,keepdims=True)
    # Normalize 
    normalized = (X-mean)/np.sqrt(var+eps)
    # Scale and Shift
    final = gamma * normalized + beta
    return final


In [134]:
np.random.seed(42)
X = np.random.randn(2, 2, 3)
gamma = np.ones(3).reshape(1, 1, -1)
beta = np.zeros(3).reshape(1, 1, -1)

In [135]:
X

array([[[ 0.49671415, -0.1382643 ,  0.64768854],
        [ 1.52302986, -0.23415337, -0.23413696]],

       [[ 1.57921282,  0.76743473, -0.46947439],
        [ 0.54256004, -0.46341769, -0.46572975]]])

In [136]:
layer_normalization(X,gamma,beta)

array([[[ 0.47373971, -1.39079736,  0.91705765],
        [ 1.41420326, -0.70711154, -0.70709172]],

       [[ 1.13192477,  0.16823009, -1.30015486],
        [ 1.4141794 , -0.70465482, -0.70952458]]])

### Positional Encoding 

In [137]:
def positional_encoding(position, d_model):
    pos = np.arange(position)[:, np.newaxis]          # (position, 1)
    i = np.arange(d_model)[np.newaxis, :]              # (1, d_model)
    angle_rates = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pos_encoding = np.zeros((position, d_model))
    pos_encoding[:, 0::2] = np.sin(angle_rates[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(angle_rates[:, 1::2])
    pos_encoding = pos_encoding[np.newaxis, :, :]
    return pos_encoding.astype(np.float32)

In [149]:
position = 2
d_model = 8
positional_encoding(position,d_model).squeeze(0).shape

(2, 8)

In [143]:
pos = np.arange(position)[:, np.newaxis]
pos,pos.shape

(array([[0],
        [1]]),
 (2, 1))

In [144]:
i = np.arange(d_model)[np.newaxis, :] 
i,i.shape

(array([[0, 1, 2, 3, 4, 5, 6, 7]]), (1, 8))

In [145]:
angle_rates = pos / np.power(10000, (2 * (i // 2)) / d_model)
angle_rates

array([[0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [1.   , 1.   , 0.1  , 0.1  , 0.01 , 0.01 , 0.001, 0.001]])

In [146]:
angle_rates.shape

(2, 8)